In [3]:
import torch
import torch.nn as nn
from torchrl.envs.utils import step_mdp, check_env_specs
from torchrl.envs.utils import ExplorationType, set_exploration_type
from torchrl.data.replay_buffers import ReplayBuffer, LazyTensorStorage
from torchrl.modules import SafeModule, ProbabilisticActor, ValueOperator
from torchrl.modules.distributions import NormalParamExtractor, TanhNormal
from torchrl.objectives.sac import SACLoss
from torchrl.objectives import SoftUpdate, group_optimizers
from torchrl.data.replay_buffers import ReplayBuffer, LazyTensorStorage
 
from state import KSPState
from qvn_model import QVNModel
 
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Using device: {device}")

Using device: cuda


For this RL problem I will be using the SAC (Soft Actor Critic) algorithm. This is an algorithm that finds the best stochastic policy: pi, and the best Q(s,a) function. We do this by approximating their values though two neural networks: theta and phi. So we estimate pi(.|s ; phi) and Q(s,a | theta). We then use a distribution to choose our actions. The second crucial part of the algorithm is that the objective function: J(theta) includes a "entropy term" which is maximized. Meaning that the model is allowed to explore the space that it is given.

Kerbal Space Program is a continious problem, I thought about making this a sequential problem by allowing "frames" where the agent would act (say every 4 frames), as in the original DQN paper. Yet Kerbal space program needs inputs that are continious such as thrust (eg sometime I need the rockets thrust to be 10% to land and not intermitent from 100% to 10%). So I discarded DQC, DDQC, and other competing algorithms for SAC. Note that the implementation of SAC that I will be using (from torchRL) was chosen due to already implementing hyperparameter optimization and two competing Q-networks, which we choose the minimum value of to minimize training time.

At saying this our primary job is to implement an enviorment using KRPC that is compatible with torchRL's enviorments. This way in the future, and maybe even within the same project we could get to the point where we can implement and compare many algorithms. A modified enviorment might still allow DDQN, PPO and more.

DAC came from both robotics, as it is generally the most popular algorithm in the area but also a series of youtube videos about training an RL agent to beat world record in the racing game: trackmania. There was an offhand mention of the algorithm in one of the videos so I decided to investigate and came upon the original DAC paper and code in github.

In [5]:
env = KSPState(
    target_orbit_altitude=80_000,
    step_interval=0.5,
    max_steps=2000,
)

N_OBS = 10
N_ACTIONS = 3
HIDDEN = 256



In [6]:
actor_net = nn.Sequential(
    nn.Linear(N_OBS, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, HIDDEN),
    nn.ReLU(),
    nn.Linear(HIDDEN, 2 * N_ACTIONS),
    NormalParamExtractor(),
)

actor_module = SafeModule(
    actor_net,
    in_keys=["observation"],
    out_keys=["loc", "scale"],
)
 
actor = ProbabilisticActor(
    module=actor_module,
    in_keys=["loc", "scale"],
    out_keys=["action"],
    spec=env.action_spec,
    distribution_class=TanhNormal,
).to(device)
 


In [7]:
qvalue = ValueOperator(
    module=QVNModel(N_OBS, N_ACTIONS, HIDDEN),
    in_keys=["observation", "action"],
).to(device)

Here we can leverage the compatibility of the state created with the torchRL package. Thus we can use the built in SAC algorithm, and specifically use the two q-networks as noted by 1801.01290v2

In [9]:
loss_module = SACLoss(
    actor_network=actor,
    qvalue_network=qvalue,
    num_qvalue_nets=2,
    alpha_init=0.2,
    target_entropy="auto",
    loss_function="smooth_l1",
)
loss_module.to(device)

target_updater = SoftUpdate(loss_module, tau=0.005)


optimizer_actor = torch.optim.Adam(
    loss_module.actor_network_params.values(True, True), lr=3e-4
)
optimizer_critic = torch.optim.Adam(
    loss_module.qvalue_network_params.values(True, True), lr=3e-4
)
optimizer_alpha = torch.optim.Adam(
    [loss_module.log_alpha], lr=3e-4
)
optimizer = group_optimizers(optimizer_actor, optimizer_critic, optimizer_alpha)
del optimizer_actor, optimizer_critic, optimizer_alpha

In [10]:
import pickle
from tensordict import TensorDict

def load_demos_into_buffer(buffer, filename="demo_buffer.pkl", n_copies=5):
    """
    Load recorded demonstrations into the replay buffer.
    n_copies: duplicate the demos to increase sampling probability.
    """
    with open(filename, "rb") as f:
        transitions = pickle.load(f)

    for _ in range(n_copies):
        for t in transitions:
            td = TensorDict({
                "observation": t["observation"],
                "action": t["action"],
                "next": TensorDict({
                    "observation": t["next_observation"],
                    "reward": t["reward"],
                    "done": t["done"],
                    "terminated": t["terminated"],
                    "truncated": t["truncated"],
                }, batch_size=[]),
            }, batch_size=[])
            buffer.add(td)

    print(f"Loaded {len(transitions)} transitions x {n_copies} copies "
          f"= {len(transitions) * n_copies} total into buffer")

The following code records demos, this was because I found that giving the AI one or two demonstrations of a successfull orbit would increase the convergance of the models significantly.

In [ ]:

import time
import torch
import pickle
from state import KSPState

def record_demo(filename="demo_buffer.pkl", step_interval=0.5):
    env = KSPState(target_orbit_altitude=80_000, step_interval=step_interval)
    td = env.reset()

    transitions = []
    print("Recording — fly the craft manually in KSP!")
    print("Press Ctrl+C to stop recording.\n")

    try:
        step = 0
        while True:

            throttle_raw = env.vessel.control.throttle
            pitch = env.vessel.flight(env.body.reference_frame).pitch
            heading = env.vessel.flight(env.body.reference_frame).heading

            prev_obs = td["observation"].clone()

            # Wait one step interval
            t_start = env.space_center.ut
            while env.space_center.ut < t_start + step_interval:
                time.sleep(0.01)


            new_obs = env._get_obs()
            intact = env._vessel_intact()


            new_throttle = env.vessel.control.throttle
            new_pitch = env.vessel.flight(env.body.reference_frame).pitch
            new_heading = env.vessel.flight(env.body.reference_frame).heading

            action = torch.tensor([
                new_throttle * 2.0 - 1.0,
                (new_pitch - pitch) / 5.0,
                ((new_heading - heading + 180) % 360 - 180) / 5.0
            ], dtype=torch.float32).clamp(-1.0, 1.0)

            reward = env._reward_function(prev_obs, new_obs, intact)

            transitions.append({
                "observation": prev_obs,
                "action": action,
                "next_observation": new_obs,
                "reward": torch.tensor([reward], dtype=torch.float32),
                "done": torch.tensor([not intact]),
                "terminated": torch.tensor([not intact]),
                "truncated": torch.tensor([False]),
            })

            td["observation"] = new_obs
            env._prev_obs = new_obs
            step += 1

            if step % 20 == 0:
                apo = new_obs[1].item()
                peri = new_obs[2].item()
                alt = new_obs[0].item()
                print(f"Step {step:4d} | alt: {alt:.3f} | apo: {apo:.3f} | peri: {peri:.3f}")

            if not intact:
                print("Vessel destroyed.")
                break

            if new_obs[2].item() >= 1.0 and new_obs[1].item() >= 1.0:
                print("ORBIT ACHIEVED!")
                break

    except KeyboardInterrupt:
        print("\nRecording stopped.")

    with open(filename, "wb") as f:
        pickle.dump(transitions, f)

    print(f"Saved {len(transitions)} transitions to {filename}")
    env.close()


if __name__ == "__main__":
    record_demo()

Recording — fly the craft manually in KSP!
Press Ctrl+C to stop recording.

Step   20 | alt: 0.006 | apo: 0.014 | peri: -7.481
Step   40 | alt: 0.032 | apo: 0.088 | peri: -7.481
Step   60 | alt: 0.082 | apo: 0.238 | peri: -7.458
Step   80 | alt: 0.158 | apo: 0.507 | peri: -7.333
Step  100 | alt: 0.265 | apo: 0.867 | peri: -7.142
Step  120 | alt: 0.386 | apo: 1.167 | peri: -6.941
Step  140 | alt: 0.507 | apo: 1.258 | peri: -6.852
Step  160 | alt: 0.619 | apo: 1.257 | peri: -6.851
Step  180 | alt: 0.722 | apo: 1.257 | peri: -6.851
Step  200 | alt: 0.815 | apo: 1.257 | peri: -6.851
Step  220 | alt: 0.898 | apo: 1.257 | peri: -6.851
Step  240 | alt: 0.973 | apo: 1.257 | peri: -6.851
Step  260 | alt: 1.038 | apo: 1.257 | peri: -6.851
Step  280 | alt: 1.095 | apo: 1.257 | peri: -6.851
Step  300 | alt: 1.143 | apo: 1.257 | peri: -6.851
Step  320 | alt: 1.183 | apo: 1.257 | peri: -6.654
Step  340 | alt: 1.213 | apo: 1.261 | peri: -6.024
Step  360 | alt: 1.235 | apo: 1.268 | peri: -5.003
Step  

In [ ]:
buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=100_000),
    batch_size=256,
)

load_demos_into_buffer(buffer, "demo_buffer.pkl", n_copies=5)

2026-03-29 10:44:29,751 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
Loaded 424 transitions x 5 copies = 2120 total into buffer


In [ ]:
BATCH_SIZE = 256
NUM_EPISODES = 500
WARMUP_STEPS = 1000
UPDATES_PER_STEP = 1
 
total_steps = 0
 
for episode in range(NUM_EPISODES):
    td = env.reset()
    episode_reward = 0.0
    episode_steps = 0
 
    while True:
        if total_steps < WARMUP_STEPS:
            td["action"] = torch.rand(3) * 2.0 - 1.0  # uniform in [-1, 1]
        else:
            td_device = td.to(device)
            with torch.no_grad(), set_exploration_type(ExplorationType.RANDOM):
                td_device = actor(td_device)
            td["action"] = td_device["action"].cpu()
            td["loc"] = td_device["loc"].cpu()
            td["scale"] = td_device["scale"].cpu()
        

        next_td = env.step(td)
 
        episode_reward += next_td["next", "reward"].item()
        episode_steps += 1
        total_steps += 1

        buffer.add(next_td)

        if total_steps >= WARMUP_STEPS and len(buffer) >= BATCH_SIZE:
            for _ in range(UPDATES_PER_STEP):
                batch = buffer.sample().to(device)
 
                loss_td = loss_module(batch)
 
                total_loss = (
                    loss_td["loss_actor"]
                    + loss_td["loss_qvalue"]
                    + loss_td["loss_alpha"]
                )
 
                optimizer.zero_grad(set_to_none=True)
                total_loss.backward()
                optimizer.step()
 
                target_updater.step()

        if next_td["next", "done"].item():
            break

        td = step_mdp(next_td)
    if episode > 0 and episode % 50 == 0:
            torch.save({
                'episode': episode,
                'total_steps': total_steps,
                'loss_module': loss_module.state_dict(),
                'optimizer': optimizer.state_dict(),
            }, f"checkpoint_ep{episode}.pt")
            print(f"Saved checkpoint_ep{episode}.pt")
    print(
        f"Episode {episode:4d} | "
        f"steps: {episode_steps:5d} | "
        f"total: {total_steps:7d} | "
        f"reward: {episode_reward:8.2f} | "
        f"orbit: {'YES' if env._orbit_achieved else 'no'}"
    )

Episode    0 | steps:   115 | total:     115 | reward:   -11.03 | orbit: no
Episode    1 | steps:    39 | total:     154 | reward:   -10.41 | orbit: no
Episode    2 | steps:   112 | total:     266 | reward:   -10.78 | orbit: no
Episode    3 | steps:   116 | total:     382 | reward:   -11.11 | orbit: no
Episode    4 | steps:    85 | total:     467 | reward:   -10.49 | orbit: no
Episode    5 | steps:    70 | total:     537 | reward:   -10.54 | orbit: no
Episode    6 | steps:    43 | total:     580 | reward:   -10.45 | orbit: no
Episode    7 | steps:    62 | total:     642 | reward:   -10.47 | orbit: no
Episode    8 | steps:    88 | total:     730 | reward:   -10.85 | orbit: no
Episode    9 | steps:    59 | total:     789 | reward:   -10.61 | orbit: no
Episode   10 | steps:    43 | total:     832 | reward:   -10.42 | orbit: no
Episode   11 | steps:    84 | total:     916 | reward:   -10.74 | orbit: no
Episode   12 | steps:    43 | total:     959 | reward:   -10.44 | orbit: no
Episode   13

In [ ]:
env.close()